# 05 · Ball: candidates → tracker → clean trajectory

The ball is 6–15 px on a 1080p wide shot and becomes 4–7 px in a 1024-px tile shrunk to 640: the COCO detector alone
finds it in a small share of the frames. `balltrack.PitchBallTracker` (default `ball.detector: tracker`):

1. **Kalman filter** in image pixels (camera motion applied when the provider gives it — PTZ);
2. **zoomed re-detection**: 320-px window around the prediction, upscaled ×2;
3. **acquisition scan** at native resolution along the grass band while lost; **motion blobs** at 640 px;
4. gates: on / just above the grass, not on a player's upper body, static objects erased after 2 s.

Then `ball.clean_ball`: in pitch metres, one candidate per frame closest to the previous accepted one (speed gate
40 m/s), gaps ≤ 1 s interpolated, Savitzky–Golay smoothing, speed. Single camera: an airborne ball projects too
far (ground-plane homography), see `docs/SOTA_REVIEW.md`.

In [ ]:
import sys, os, time, json, warnings
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "football_eval").is_dir())
sys.path.insert(0, str(ROOT))
warnings.filterwarnings("ignore")
import cv2, numpy as np, pandas as pd, matplotlib.pyplot as plt
pd.set_option("display.precision", 3); pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 60)
plt.rcParams.update({"figure.dpi": 90, "axes.spines.top": False, "axes.spines.right": False})
from football_eval import nbtools as nbt

P = nbt.load_params()            # notebooks/pipeline/params.yaml (or $FVE_NB_PARAMS)
ctx = nbt.build_context(P)       # config <- match setup, restricted to the window
cfg = ctx.cfg

In [ ]:
from football_eval.detection import build_detector
from football_eval.balltrack import PitchBallTracker
from football_eval.ball import clean_ball
from football_eval.people import grass_top, grass_roi
from football_eval.pitch import Pitch, draw_pitch
pitch = Pitch(cfg.pitch.length, cfg.pitch.width)
dets = nbt.load_stage(ctx, "detections"); hom = nbt.load_stage(ctx, "homography").set_index("frame")
tracks = nbt.load_stage(ctx, "tracks", required=False)
detector = build_detector(cfg)

## 1 · What the plain detector gives (from 02)

In [ ]:
bd = dets[dets.cls == "ball"]
nfr = dets.frame.nunique()
print(f"ball candidates (conf >= {cfg.detection.conf_ball}) in {bd.frame.nunique()}/{nfr} frames ({bd.frame.nunique() / max(nfr, 1):.0%}); "
      f"{(bd.groupby('frame').size() > 1).sum()} frames with >1 candidate")
ims, titles = [], []
for r in bd.sort_values("conf", ascending=False).head(10).itertuples():
    im = nbt.read_frame_idx(ctx.video, int(r.frame))
    cx, cy = (r.x0 + r.x1) / 2, (r.y0 + r.y1) / 2
    ims.append(nbt.crop(im, [cx - 40, cy - 40, cx + 40, cy + 40], pad=0, size=160)); titles.append(f"f{int(r.frame)} {r.conf:.2f}")
nbt.grid(ims, titles, cols=5, size=2.4) if ims else print("no ball candidate from the detector in this window");

## 2 · The tracker, frame by frame
The detector's ball threshold is lowered to `ball.tracker_conf` (weak hits are gated by the track), exactly as in the integrated loop. `source` tells which mechanism measured the ball.

In [ ]:
if hasattr(detector, "cb"):
    detector.cb = min(detector.cb, cfg.ball.tracker_conf)
sc_ = (cfg.video.resize_width / ctx.info.width) if cfg.video.resize_width else 1.0
bt = PitchBallTracker(detector, cfg.video.analysis_fps, int(cfg.video.resize_width or ctx.info.width), scan_every=cfg.ball.scan_every,
                      ignore=[[v * sc_ for v in r] for r in (cfg.video.ignore_regions or [])])
hits, frames_seen, keep_last = [], [], {}
with nbt.Timer() as tm:
    for n, (f, t, img) in enumerate(nbt.frames_in_window(ctx)):
        gtop = grass_top(img)
        det = detector(img, roi=grass_roi(gtop, img.shape) if cfg.detection.pitch_roi else None)
        people = det.of("player", "goalkeeper", "referee"); b = det.of("ball")
        M = None
        if f in hom.index and "m00" in hom.columns and np.isfinite(hom.loc[f, "m00"]):
            M = hom.loc[f, [f"m{i}{j}" for i in range(3) for j in range(3)]].to_numpy(float).reshape(3, 3)
        hit = bt.step(img, b.xyxy, b.conf, people.xyxy, gtop, M)
        if bt.erase:                                   # the last frames' "ball" was a static object
            hits = [h for h in hits if h[-1] < n - bt.erase]; bt.erase = 0
        if hit is not None:
            hits.append((f, t, *hit, n))
        frames_seen.append((f, t))
        if n in (5, 60, 120): keep_last[n] = (f, img.copy(), hit)
H_ = pd.DataFrame(hits, columns=["frame", "t", "u", "v", "radius", "source", "k"])
print(f"{len(frames_seen)} frames in {tm.s:.0f}s ({len(frames_seen) / tm.s:.1f} fps incl. detector) | ball measured in {len(H_)} frames "
      f"({len(H_) / len(frames_seen):.0%})")
pd.Series(bt.stats).to_frame("count").T

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(12, 4.6), sharex=True)
for s_, g in H_.groupby("source"):
    axes[0].plot(g.t, g.u, ".", label=s_); axes[1].plot(g.t, g.v, ".", label=s_)
axes[0].set_ylabel("u (px)"); axes[1].set_ylabel("v (px)"); axes[0].legend(ncol=5, fontsize=8); axes[1].set_xlabel("t (s)")
axes[0].set_title("measured ball position by source");

## 3 · Pitch coordinates and cleaning

In [ ]:
raw = []
for r in H_.itertuples():
    Hm = nbt.homography_from_row(hom.loc[r.frame]) if r.frame in hom.index else None
    x, y = Hm.to_pitch(np.array([[r.u, r.v]]))[0] if Hm is not None else (np.nan, np.nan)
    conf = {"full": 0.6, "zoom": 0.5, "scan": 0.45, "motion": 0.3, "blob": 0.2}.get(r.source, 0.3)
    raw.append((r.frame, r.t, conf, r.u, r.v, x, y))
ball_raw = pd.DataFrame(raw, columns=["frame", "t", "conf", "u", "v", "x", "y"])
fr = pd.DataFrame(frames_seen, columns=["frame", "t"])
ball = clean_ball(ball_raw, fr.frame.to_numpy(), fr.t.to_numpy(), cfg.ball.max_speed_ms, cfg.ball.max_gap_s, cfg.ball.smooth_window)
print(f"visible {ball.visible.mean():.0%} | interpolated {ball.interp.mean():.0%} | usable {ball.valid.mean():.0%} | "
      f"rejected by the speed gate: {ball_raw.x.notna().sum() - ball.visible.sum()}")
fig, axes = plt.subplots(1, 2, figsize=(15, 3.8))
axes[0].plot(ball_raw.t, ball_raw.x, ".", c="0.6", label="raw x"); axes[0].plot(ball.t, ball.x, label="clean x")
axes[0].plot(ball_raw.t, ball_raw.y, ".", c="0.8", label="raw y"); axes[0].plot(ball.t, ball.y, label="clean y")
axes[0].legend(fontsize=8); axes[0].set_title("ball (m)")
axes[1].plot(ball.t, ball.speed); axes[1].set_title("ball speed (m/s)"); axes[1].set_ylim(0, 45);

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5.4))
draw_pitch(ax, pitch)
v = ball[ball.valid]
ax.scatter(v.x, v.y, c=v.t, cmap="viridis", s=10)
if tracks is not None:
    q = tracks[(tracks.frame == tracks.frame.median()) & tracks.x.notna()]
    ax.scatter(q.x, q.y, c=["C3" if t_ == 0 else "C0" if t_ == 1 else "0.5" for t_ in q.team], s=25)
ax.set_title("ball trajectory over the window (colour = time)");

In [ ]:
nbt.save_stage(ctx, "ball_raw", ball_raw); nbt.save_stage(ctx, "ball", ball);